In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d

from astropy.io import fits
from astropy.convolution import convolve, convolve_fft
import scipy.io as sio
from scipy import signal
from pathlib import Path
from scipy import ndimage
import scipy.ndimage as ndi
import imageio.v2 as imageio
from skimage import filters, morphology, measure, segmentation

ModuleNotFoundError: No module named 'numpy'

In [ ]:
image_path = Path(r"..\Images\follicule.bmp")
image = plt.imread(image_path)
plt.imshow(image, cmap="gray")
plt.show()

B = image[:,:,2];
antrum = B > 220;
L = morphology.label(antrum, connectivity =2) ;
antrum = L == L [300,300];
antrum = ndimage.morphology. binary_fill_holes (antrum);
plt .imshow(antrum);
plt.title('Antrum')
plt.show();


In [ ]:
se40 = morphology.disk(40) ;
theca = morphology.binary_dilation(antrum, footprint = se40);
theca = theca & ~antrum;
plt.imshow(theca);
plt.title ( 'Theca')
plt.show()


In [ ]:
vascularization = B < 140;
vascularization = vascularization & theca
plt.imshow(vascularization ) ;
plt.title ( 'vascularization' )
plt.show();

In [ ]:
circles = plt.imread(r"C:\Users\Hippolyte\Desktop\TB Images\Images\circles.TIF")


In [ ]:
def rmax(image):
    """Own version of regional maximum for integer images."""
    image = image.astype(float)
    image = image / np.max(image) * (2 ** 31)
    image = image.astype("int32")
    h = 1
    rec = morphology.reconstruction(image, image + h)
    maxima = image + h - rec
    return maxima > 0

A = imageio.imread(r"C:\Users\Hippolyte\Desktop\TB Images\Images\circles.TIF")
A = A > 0

dm = ndimage.distance_transform_edt(A)
local_maxi = rmax(dm)

markers = ndi.label(local_maxi, np.ones((3, 3)))[0]
W = segmentation.watershed(-dm, markers, watershed_line=True)
B = A & (W == 0)
separation = ndi.label(B, np.ones((3, 3)))[0]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(A, cmap="gray")
axes[0].set_title("circles")
axes[1].imshow(dm, cmap="magma")
axes[1].set_title("distance map")
axes[2].imshow(W == 0, cmap="gray")
axes[2].set_title("watershed lines")
axes[3].imshow(separation, cmap="nipy_spectral")
axes[3].set_title("separated labels")
for ax in axes:
    ax.axis("off")
plt.tight_layout()